# Pathfinder basic control and link checks

This notebook uses `pathfinder_control.py` and a selected mapping YAML from the same directory. Both Pathfinder and Eliminator mappings are supported. Run cells sequentially. Sections 1-2 are offline. Sections 3 onward access hardware. This notebook does not start injection or acquisition.

Use the existing Tamalero hardware environment for hardware sections. The imported ETROC driver must support `i2c_master`. Stop any other acquisition before initializing or changing readout masks. Restart the kernel after changing the control module. Changing the selection requires rerunning initialization and all following checks.

Status messages use colors. BL/NW uses one total progress bar with chip identity and failure count, followed by per-chip summaries. Driver stdout is retained in each scan record; no per-pixel values are printed. tqdm is required; tqdm.auto selects the available progress-bar backend.

## 0. Set repository paths for GUI execution - offline
Edit TAMALERO_ROOT below to the actual repository root on the kernel computer, then run this first code cell. Select the same Python kernel as the existing hardware test notebook. This sets Python search paths, TAMALERO_BASE and the working directory; it does not install dependencies or run setup.sh. Restart the kernel when switching driver repositories.

The working directory stays at the repository root for legacy configs/ references. Mapping and result paths point explicitly into pathfinder/.

In [4]:
import os
import sys
from pathlib import Path

# Replace this with the repository root on the computer running this kernel.
# The root contains configs/, address_table/, tamalero/, and pathfinder/.
# TAMALERO_ROOT = Path("/home/roy/etroc2_pathfinder/tamalero").expanduser().resolve()
TAMALERO_ROOT = Path(r"D:\XingHUANG\ETROC2\ETROC2_PathFinder\tamalero").expanduser().resolve()

if not (TAMALERO_ROOT / "tamalero" / "ETROC.py").is_file():
    raise FileNotFoundError(f"Incorrect repository root: {TAMALERO_ROOT}")

if not (TAMALERO_ROOT / "pathfinder" / "pathfinder_control.py").is_file():
    raise FileNotFoundError(f"Missing pathfinder_control.py in {TAMALERO_ROOT / 'pathfinder'}")

# Restart the kernel before using this cell to switch driver repositories.
sys.path.insert(0, str(TAMALERO_ROOT))
sys.path.insert(0, str(TAMALERO_ROOT / "pathfinder"))
os.environ["TAMALERO_BASE"] = str(TAMALERO_ROOT)
# Legacy drivers resolve configs/ relative to the repository root.
os.chdir(TAMALERO_ROOT)

print("Python:", sys.executable)
print("Repository:", TAMALERO_ROOT)
print("Working directory:", Path.cwd())

Python: c:\Program Files\Python\Python311\python.exe
Repository: D:\XingHUANG\ETROC2\ETROC2_PathFinder\tamalero
Working directory: D:\XingHUANG\ETROC2\ETROC2_PathFinder\tamalero


## 1. Test selection and connection parameters - offline
Edit the module/Hybrid selection here. A module mapped to `"all"` selects all four Hybrids. The initial selection is an example, not a hardware requirement.

In [18]:
from html import escape
from IPython.display import HTML, display
from tqdm.auto import tqdm


def show_status(message, level="info"):
    colors = {"success": "#16803c", "warning": "#a66b00",
              "error": "#c62828", "info": "#2767a0"}
    color = colors[level]
    display(HTML(f'<div style="color:{color};font-weight:600">'
                 f'{escape(str(message))}</div>'))


from pathlib import Path
import os
import inspect

from pathfinder_control import (
    load_mapping, expand_selection, build_elinks, build_readout_masks,
    initialize_etrocs, InitializationError,
    check_connections, check_links, configure_readout_links, build_trigger_mask,
)

MAPPING_PATH = TAMALERO_ROOT / "pathfinder" / "pathfinder_mapping.yaml"
ENABLE_SELECTION = {1: "all", 3: [1]}
# ENABLE_SELECTION = {m: "all" for m in range(1, 8)}
# For Eliminator, replace both settings above with:
# MAPPING_PATH = TAMALERO_ROOT / "pathfinder" / "eliminator_mapping.yaml"
# ENABLE_SELECTION = {1: [1], 3: [1], 4: [1]}
# ENABLE_SELECTION = {m: "all" for m in range(1, 5)}

KCU_IP = "192.168.0.10"
READOUTBOARD_ID = 0
READOUTBOARD_CONFIG = "default"
ELINK_WIDTH = 0x3  # Retained from the existing test notebook.
TAMALERO_BASE = os.environ.get("TAMALERO_BASE")

basic_checks_passed = False

## 2. Load and preview the selected wiring - offline
The YAML remains the source of wiring information. A data route may use a different lpGBT than its I2C route.

In [19]:
mapping = load_mapping(MAPPING_PATH)
ids = expand_selection(mapping, ENABLE_SELECTION)
USE_TRIG = "trig" in mapping["lpgbts"]
basic_checks_passed = False

print(f"Selected {len(ids)} ETROCs: {ids}")
for identity in ids:
    cfg = mapping["etrocs"][identity]
    bus = cfg["i2c"]
    print(f"{identity}: I2C={bus['lpgbt']}/M{bus['channel']}/"
          f"0x{bus['address']:02X}, elinks={build_elinks(mapping, identity)}")

preview_masks = build_readout_masks(mapping, ids)
for bank, mask in preview_masks.items():
    print(f"Bank {bank} disable mask: 0x{mask:07X}")

Selected 5 ETROCs: ['D1H1', 'D1H2', 'D1H3', 'D1H4', 'D3H1']
D1H1: I2C=daq/M0/0x60, elinks={0: [22], 1: []}
D1H2: I2C=daq/M0/0x61, elinks={0: [20], 1: []}
D1H3: I2C=daq/M0/0x62, elinks={0: [18], 1: []}
D1H4: I2C=daq/M0/0x63, elinks={0: [16], 1: []}
D3H1: I2C=trig/M1/0x60, elinks={0: [], 1: [10]}
Bank 0 disable mask: 0xFAAFFFF
Bank 1 disable mask: 0xFFFFBFF


### 2.1. Trigger Selection and Offline Mask Check

Select which ETROCs participate in self-trigger generation, independently of the readout selection.

In [21]:
# Select trigger sources independently from readout selection.
TRIGGER_SELECTION = {1: [1], 3: [1]}
TRIGGER_COMBINATION_LOGIC = 0  # 0 = OR; 1 = AND

trigger_ids = expand_selection(mapping, TRIGGER_SELECTION)

if not set(trigger_ids).issubset(set(ids)):
    raise ValueError("Trigger sources must be included in ENABLE_SELECTION")

preview_trigger_mask = build_trigger_mask(mapping, trigger_ids)

print("Trigger ETROCs:", trigger_ids)
print(f"Trigger mask: 0x{preview_trigger_mask:07X}")

Trigger ETROCs: ['D1H1', 'D3H1']
Trigger mask: 0x0080800


## 3. Initialize KCU - hardware
The loopback test writes and reads the existing LOOPBACK register. If it fails, do not continue to the next cell.

In [ ]:
from tamalero.KCU import KCU
from tamalero.ReadoutBoard import ReadoutBoard
from tamalero.ETROC import ETROC

if "i2c_master" not in inspect.signature(ETROC.__init__).parameters:
    raise RuntimeError("The imported ETROC driver does not support i2c_master")
if not TAMALERO_BASE:
    raise RuntimeError("Set TAMALERO_BASE to the existing Tamalero project root")

xml_path = Path(TAMALERO_BASE) / "address_table" / "generic" / "etl_test_fw.xml"
if not xml_path.is_file():
    raise FileNotFoundError(xml_path)

print(f"ETROC driver: {inspect.getfile(ETROC)}")
print(f"Address table: {xml_path}")
kcu = KCU(
    name="kcu",
    ipb_path=f"chtcp-2.0://localhost:10203?target={KCU_IP}:50001",
    adr_table=str(xml_path),
)
kcu.status()
loopback_value = 0xABCD1234
kcu.write_node("LOOPBACK.LOOPBACK", loopback_value)
received = kcu.read_node("LOOPBACK.LOOPBACK").value()
if received != loopback_value:
    raise RuntimeError(f"Loopback failed: wrote 0x{loopback_value:X}, read 0x{received:X}")
show_status("KCU loopback passed", "success")

## 4. Initialize declared lpGBTs and mask their data links - hardware
DAQ is always initialized. TRIG is initialized only when declared in the mapping, regardless of the current chip selection. The existing ReadoutBoard constructor configures hardware and may initially enable links. Declared banks are masked immediately after it returns. A constructor failure can leave partial hardware state; stop and investigate before continuing.

DAQ-only mode does not instantiate TRIG or access its mask through the new control utilities. However, the existing ReadoutBoard constructor and ETROC driver may still access SLAVE FPGA registers. A firmware image without those registers requires a separate driver compatibility review; this notebook does not modify those drivers.

The current driver uses DAQ M0 / 0x72 to reach TRIG. The YAML records that connection but does not reconfigure the driver's upstream route. The check below detects a mismatch with this known driver configuration.

In [ ]:
USE_TRIG = "trig" in mapping["lpgbts"]
if USE_TRIG:
    upstream = mapping["lpgbts"]["trig"]["control"]
    if (upstream["via"], upstream["channel"], upstream["address"]) != ("daq", 0, 0x72):
        raise RuntimeError("The YAML TRIG route differs from the current DAQ M0 / 0x72 driver route")

basic_checks_passed = False
chips = {}
initialized_ids = ()
etroc_register_checks_done = False
etroc_power_checks_done = False
blnw_scan_passed = False
rb = ReadoutBoard(
    rb=READOUTBOARD_ID, kcu=kcu, config=READOUTBOARD_CONFIG,
    trigger=USE_TRIG, verbose=True,
)
configure_readout_links(rb, mapping, [])
if USE_TRIG and (not rb.trigger or getattr(rb, "TRIG_LPGBT", None) is None):
    raise RuntimeError("TRIG lpGBT was not initialized; readout links remain masked")

# Retain the TRIG clock configuration verified during hardware testing.
if USE_TRIG:
    rb.TRIG_LPGBT.configure_clocks(0x0fffffff)
    show_status("TRIG ECLK0-27 configured; verify readback below", "info")

width_node = f"READOUT_BOARD_{rb.rb}.ELINK_WIDTH"
rb.kcu.write_node(width_node, ELINK_WIDTH)
width = rb.kcu.read_node(width_node).value()
if width != ELINK_WIDTH:
    raise RuntimeError(f"ELINK_WIDTH mismatch: expected {ELINK_WIDTH}, received {width}")
print(f"Initialized controllers: {list(mapping['lpgbts'])}; declared banks are masked")

### DAQ lpGBT link status and FEC counters
Read the DAQ link status and FEC counters after board initialization. Stop if the DAQ link is not good. Preserve the initial counters in `fec_errors_before_reset`, then reset the counters using the existing board method. This reset changes hardware counters and may affect all links handled by that method.
A zero counter immediately after reset does not demonstrate error-free operation over time. Read the counters again after a test interval. This cell does not validate ETROC data or TRIG link health.

In [ ]:
print("Checking DAQ LpGBT base configuration:")
daq_link_good = rb.DAQ_LPGBT.link_status()
show_status(f"DAQ LpGBT Link Good: {daq_link_good}", "success" if daq_link_good else "error")

fec_errors_before_reset = rb.get_FEC_error_count()
print(f"FEC Errors: DAQ LpGBT = {fec_errors_before_reset.get('DAQ', 'N/A')}")

if not daq_link_good:
    basic_checks_passed = False
    raise RuntimeError("DAQ lpGBT link is not good; stop before ETROC initialization")

# Preserve the observed counters above before starting a new counting interval.
rb.reset_FEC_error_count()

### Clock register readback - read only
Run after ReadoutBoard initialization and before ETROC initialization. Read ECLK0-27 on DAQ and on TRIG when its object exists. No clock settings are written. Results are retained in `clock_readback`.
This is diagnostic output, not a pass/fail clock configuration test. The existing `configure_clocks()` writes FREQ=1 and DRIVESTRENGTH=4; other initialization paths may use different settings. Readback does not identify the writer or verify a physical clock waveform. ECLK28 is outside this check.

In [ ]:
def show_clock_config(lpgbt, name):
    results = []
    print(f"\n{name} clock configuration")
    for channel in range(28):
        prefix = f"LPGBT.RWF.EPORTCLK.EPCLK{channel}"
        result = {"channel": channel, "frequency": None,
                  "drive_strength": None, "error": None}
        try:
            result["frequency"] = lpgbt.rd_reg(f"{prefix}FREQ")
            result["drive_strength"] = lpgbt.rd_reg(f"{prefix}DRIVESTRENGTH")
            print(f"ECLK{channel:02d}: FREQ={result['frequency']}, "
                  f"DRIVESTRENGTH={result['drive_strength']}")
        except Exception as exc:
            result["error"] = str(exc)
            print(f"ECLK{channel:02d}: read failed: {exc}")
        results.append(result)
    return results


clock_readback = {"daq": show_clock_config(rb.DAQ_LPGBT, "DAQ")}
trig_lpgbt = getattr(rb, "TRIG_LPGBT", None)
if trig_lpgbt is not None:
    clock_readback["trig"] = show_clock_config(trig_lpgbt, "TRIG")
else:
    print("\nTRIG object is unavailable; skipping TRIG clock readback.")

clock_read_errors = sum(
    result["error"] is not None
    for results in clock_readback.values()
    for result in results
)
show_status(f"Clock register read errors: {clock_read_errors}", "success" if clock_read_errors == 0 else "error")

## 5. Initialize selected ETROCs - hardware
Construction follows the existing ETROC driver defaults and may configure registers or perform driver-defined resets. The later configuration cells pulse PS_CapRst, read key registers and set High power mode. The later BL/NW cell also runs auto_threshold_scan using the legacy arguments. Position IDs do not program the hardware chip_id.

In [ ]:
basic_checks_passed = False
chips = {}
initialized_ids = ()
etroc_register_checks_done = False
etroc_power_checks_done = False
blnw_scan_passed = False
configure_readout_links(rb, mapping, [])
try:
    chips = initialize_etrocs(rb, mapping, ids, strict=False, verbose=False)
except InitializationError as exc:
    chips = exc.initialized_chips
    print(f"Failed ETROC: {exc.failed_id}; completed: {list(chips)}")
    raise
initialized_ids = tuple(ids)
show_status(f"Initialized {len(chips)} ETROCs: {list(chips)}", "success")

## 6. Check I2C communication - hardware
A successful connectivity check is not a chip identity test. Failure prevents this cell from completing; do not continue until resolved.

In [ ]:
basic_checks_passed = False
if tuple(ids) != initialized_ids or list(chips) != ids:
    raise RuntimeError("Selection changed or initialization is incomplete; rerun initialization")
connection_results = check_connections(chips)
for identity, result in connection_results.items():
    show_status(f"{identity}: connected={result['connected']}, error={result['error']}", "success" if result["connected"] else "error")
if not all(result["connected"] for result in connection_results.values()):
    raise RuntimeError("One or more ETROCs failed the I2C check")

### 6a. Configure ETROC reset and read key registers

Pulse PS_CapRst (1 then 0) on each selected ETROC and read disScrambler, controllerState, pllUnlockCount and L1Adelay. Legacy expectations (1 and 11) produce warnings, not communication-failure claims. Exceptions stop this cell; completed hardware changes are not rolled back. Readout stays masked.

In [ ]:
basic_checks_passed = False
etroc_register_checks_done = False
etroc_power_checks_done = False
blnw_scan_passed = False
if not ids or tuple(ids) != initialized_ids or list(chips) != ids:
    raise RuntimeError("Selection is empty or initialization is incomplete; rerun initialization")
configure_readout_links(rb, mapping, [])
etroc_register_results = {}
for identity, etroc in chips.items():
    values = {}
    etroc_register_results[identity] = values
    try:
        if not etroc.is_connected():
            raise RuntimeError("ETROC is not responding")
        etroc.wr_reg("PS_CapRst", 1)
        etroc.wr_reg("PS_CapRst", 0)
        for register in ("disScrambler", "controllerState", "pllUnlockCount", "L1Adelay"):
            values[register] = etroc.rd_reg(register)
        print(f"{identity}: {values}")
        if values["disScrambler"] != 1:
            show_status(f"{identity}: disScrambler differs from the legacy expected value 1", "warning")
        if values["controllerState"] != 11:
            show_status(f"{identity}: controllerState differs from the legacy expected value 11", "warning")
    except Exception as exc:
        values["error"] = str(exc)
        raise RuntimeError(f"ETROC register configuration failed for {identity}; readout remains masked") from exc
etroc_register_checks_done = True

### 6b. Set and verify High power mode

Broadcast High power mode to each selected ETROC, then sample pixel (8, 8) on each device. This sample does not verify every pixel. A write failure, read failure or mode mismatch stops the cell. Successful completion is required before enabling data links.

In [ ]:
basic_checks_passed = False
etroc_power_checks_done = False
blnw_scan_passed = False
if tuple(ids) != initialized_ids or list(chips) != ids:
    raise RuntimeError("Selection changed or initialization is incomplete")
if not etroc_register_checks_done:
    raise RuntimeError("Complete the ETROC register configuration cell first")
configure_readout_links(rb, mapping, [])
etroc_power_results = {}
for identity, etroc in chips.items():
    try:
        etroc.set_power_mode(mode="high", row=0, col=0, broadcast=True)
    except Exception as exc:
        raise RuntimeError(f"High power configuration failed for {identity}; readout remains masked") from exc

for identity, etroc in chips.items():
    try:
        mode = etroc.get_power_mode(row=8, col=8)
        etroc_power_results[identity] = mode
        print(f"{identity}: power mode at pixel (8, 8) = {mode}")
        if mode != "high":
            raise RuntimeError(f"Expected high power mode, received {mode!r}")
    except Exception as exc:
        raise RuntimeError(f"Power mode verification failed for {identity}; readout remains masked") from exc
etroc_power_checks_done = True
show_status("High power mode configured; sampled pixel readbacks passed", "success")

### 6c. BL/NW scan settings

Default: 8 pixels per selected ETROC (rows 0, 15; columns 0, 7, 8, 15). Select `all` for 256 pixels per ETROC.

In [ ]:
# Choose "sample" for 8 pixels or "all" for the full 16 x 16 matrix.
BLNW_SCAN_MODE = "sample"
BLNW_SAVE_RESULTS = True  # False keeps results in memory without writing files.
BLNW_OUTPUT_DIR = TAMALERO_ROOT / "pathfinder" / "results" / "blnw" 

### 6d. Run BL/NW calibration and save results

This cell changes hardware through auto_threshold_scan, using the original arguments. It does not promise restoration of the pre-scan register state. Each pixel is attempted once; no automatic GPIO reset, PLL calibration or bitslip is performed. Ordinary scan exceptions are recorded and remaining pixels are attempted. Failures use None, not zero. When BLNW_SAVE_RESULTS is True, results are checkpointed to a new pickle file after each pixel; Interrupted scans retain completed records with completed=False in memory, and on disk only when saving is enabled. Set the save switch before starting the scan; its value is captured for that run. Run this cell again to start a fresh scan, not resume. All scans must finish without exceptions before the link-enable cell can proceed; this does not impose physics acceptance limits on BL/NW values.

In [ ]:
import pickle
import io
from contextlib import redirect_stdout
from datetime import datetime

def build_blnw_pixels(mode):
    if mode == "sample":
        return [(row, col) for row in (0, 15) for col in (0, 7, 8, 15)]
    if mode == "all":
        return [(row, col) for row in range(16) for col in range(16)]
    raise ValueError("BLNW_SCAN_MODE must be 'sample' or 'all'")


basic_checks_passed = False
blnw_scan_passed = False
scan_pixels = build_blnw_pixels(BLNW_SCAN_MODE)
if not ids or tuple(ids) != initialized_ids or list(chips) != ids:
    raise RuntimeError("Selection is empty or initialization is incomplete")
if not etroc_register_checks_done or not etroc_power_checks_done:
    raise RuntimeError("Complete ETROC register and power configuration first")
configure_readout_links(rb, mapping, [])
blnw_save_enabled = BLNW_SAVE_RESULTS
blnw_output_path = None
if blnw_save_enabled:
    BLNW_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    blnw_output_path = BLNW_OUTPUT_DIR / (
        "blnw_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f") + ".pkl"
    )
baseline_dict = {identity: {} for identity in ids}
nw_dict = {identity: {} for identity in ids}
failed_pixels = {identity: [] for identity in ids}
blnw_results = {
    "baseline": baseline_dict,
    "NW": nw_dict,
    "chip_name": list(ids),
    "failed_pixels": failed_pixels,
    "records": [],
    "scan_mode": BLNW_SCAN_MODE,
    "pixels": scan_pixels,
    "mapping_path": str(MAPPING_PATH),
    "started_at": datetime.now().isoformat(),
    "completed": False,
}


def save_blnw_results():
    if not blnw_save_enabled:
        return
    temporary_path = blnw_output_path.with_suffix(".tmp")
    with temporary_path.open("wb") as handle:
        pickle.dump(blnw_results, handle)
    temporary_path.replace(blnw_output_path)


save_blnw_results()
print(f"Scanning {len(scan_pixels)} pixels per ETROC on {len(chips)} ETROCs")
if blnw_save_enabled:
    print(f"Results: {blnw_output_path.resolve()}")
else:
    print("Local saving is disabled; results remain in blnw_results")

scan_failure_count = 0
try:
    for identity, etroc in chips.items():
        progress = tqdm(total=len(scan_pixels), desc=f"BL/NW scan: {identity}",
                        unit="pixel", leave=True)
        try:
            for row, col in scan_pixels:
                record = {"chip": identity, "row": row, "col": col,
                          "baseline": None, "noise_width": None, "error": None}
                driver_output = io.StringIO()
                progress.set_postfix(row=row, col=col,
                                     failed=len(failed_pixels[identity]), refresh=False)
                try:
                    # Retain unsolicited driver output in the result, not in the cell output.
                    with redirect_stdout(driver_output):
                        baseline, noise_width = etroc.auto_threshold_scan(
                            row=row, col=col, broadcast=False,
                            offset=1023, use=False, verbose=False,
                        )
                    if baseline is None or noise_width is None:
                        raise RuntimeError("Scan returned an incomplete result")
                    record.update(baseline=baseline, noise_width=noise_width)
                except Exception as exc:
                    record["error"] = str(exc)
                    failed_pixels[identity].append((row, col))
                    scan_failure_count += 1
                record["driver_output"] = driver_output.getvalue()
                record["timestamp"] = datetime.now().isoformat()
                baseline_dict[identity][(row, col)] = record["baseline"]
                nw_dict[identity][(row, col)] = record["noise_width"]
                blnw_results["records"].append(record)
                save_blnw_results()
                progress.set_postfix(failed=len(failed_pixels[identity]), refresh=False)
                progress.update(1)
        finally:
            progress.close()
    blnw_results["completed"] = True
finally:
    blnw_results["last_saved_at"] = datetime.now().isoformat()
    save_blnw_results()

failure_count = sum(len(pixels) for pixels in failed_pixels.values())
show_status(f"BL/NW scan finished: {len(blnw_results['records'])} pixels, "
            f"{failure_count} failed", "error" if failure_count else "success")
for identity in ids:
    count = len(failed_pixels[identity])
    show_status(f"{identity}: {len(scan_pixels) - count}/{len(scan_pixels)} pixels passed",
                "warning" if count else "success")
if failure_count:
    raise RuntimeError(f"{failure_count} pixels failed; inspect failed_pixels and blnw_results; readout remains masked")
blnw_scan_passed = True

### 6e. Display fixed 16 x 16 BL/NW maps
Each selected ETROC has a baseline map and a noise-width map. Columns are horizontal and rows are vertical, with row 0 at the bottom. Unscanned pixels and failed measurements remain blank; valid zero values are shown. Plotting works whether local saving is enabled or disabled. After a scan error or interruption, this cell can be run manually to inspect available results. Images are displayed only, not automatically saved.
To plot an earlier locally saved result, load only a trusted pickle file into `blnw_results` using `pickle.load`, then rerun this cell. Pickle files must not be loaded from untrusted sources.

For results recorded with scan_mode="all", also display separate BL and NW histograms for each ETROC. Only finite measured values contribute; valid zero values are included. Each histogram shows the valid pixel count, mean and population standard deviation. Partial or failed full-matrix scans show only available measurements. Sample scans do not produce histograms. The mode comes from the results, including when reloading a saved file.

NW map annotations and colorbar ticks use integers. Histogram x-axis limits are fixed: BL 0-1023 and NW 0-17. NW uses one bin per integer value. Mean and standard deviation remain fractional statistics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, FormatStrFormatter
from datetime import datetime


def format_scan_time(results):
    """Use the saved scan start time; never substitute the plotting time."""
    timestamp = results.get("started_at")
    if not timestamp:
        return "Scan time unavailable"
    try:
        return "Scan: " + datetime.fromisoformat(str(timestamp)).isoformat(sep=" ", timespec="seconds")
    except (TypeError, ValueError):
        return "Scan: " + str(timestamp)


def build_blnw_matrix(values):
    matrix = np.full((16, 16), np.nan)
    for (row, col), value in values.items():
        if value is not None:
        # if value is not None and value != 0:
            matrix[row, col] = float(value)
    return matrix


def plot_blnw_maps(results):
    figures = []
    for identity in results["chip_name"]:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
        for ax, key, title in zip(axes, ("baseline", "NW"), ("Baseline", "Noise width")):
            matrix = build_blnw_matrix(results[key][identity])
            cmap = plt.get_cmap("viridis").copy()
            cmap.set_bad("white")
            view = ax.imshow(np.ma.masked_invalid(matrix), origin="lower",
                             cmap=cmap, interpolation="nearest")
            # ax.set_title(f"{identity} - {title}")
            ax.text(0.01, 1.00, "CMS", transform=ax.transAxes,
                    fontsize=13, fontweight="bold", va="bottom")
            ax.text(0.11, 1.00, "ETL ETROC", transform=ax.transAxes,
                    fontsize=10, style="italic", va="bottom")
            ax.text(1.0, 1.025, f"{identity}-{title} 2D", transform=ax.transAxes,
                    fontsize=9, ha="right", va="bottom")
            ax.text(1.0, 1.00, format_scan_time(results), transform=ax.transAxes,
                    fontsize=9, ha="right", va="bottom")
            ax.set_xlabel("Column")
            ax.set_ylabel("Row")
            ax.set_xticks(range(16))
            ax.set_yticks(range(16))
            ax.set_xticks(np.arange(-0.5, 16, 1), minor=True)
            ax.set_yticks(np.arange(-0.5, 16, 1), minor=True)
            ax.grid(which="minor", color="lightgray", linewidth=0.4)
            ax.tick_params(which="minor", bottom=False, left=False)
            for row, col in np.argwhere(np.isfinite(matrix)):
                value = matrix[row, col]
                rgba = view.cmap(view.norm(value))
                brightness = sum(weight * component for weight, component in
                                 zip((0.299, 0.587, 0.114), rgba[:3]))
                label = f"{value:.0f}" if key == "NW" else f"{value:g}"
                ax.text(col, row, label, ha="center", va="center",
                        fontsize=6, color="black" if brightness > 0.5 else "white")
            if np.isfinite(matrix).any():
                colorbar = fig.colorbar(view, ax=ax, label="DAC code")
                if key == "NW":
                    colorbar.locator = MaxNLocator(integer=True)
                    colorbar.formatter = FormatStrFormatter("%.0f")
                    colorbar.update_ticks()
            else:
                ax.set_title(f"{identity} - {title} (no valid results)")
        # fig.suptitle("Unscanned or failed pixels are blank")
        figures.append(fig)
        plt.show()
    return figures


def plot_blnw_histograms(results):
    figures = []
    if results.get("scan_mode") != "all":
        return figures
    for identity in results["chip_name"]:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
        for ax, key, title in zip(axes, ("baseline", "NW"), ("Baseline", "Noise width")):
            matrix = build_blnw_matrix(results[key][identity])
            values = matrix[np.isfinite(matrix) & (matrix!=0)]
            # ax.set_title(f"{identity} - {title} (valid pixels: {values.size}/256)")
            ax.text(0.01, 1.00, "CMS", transform=ax.transAxes,
                    fontsize=13, fontweight="bold", va="bottom")
            ax.text(0.11, 1.00, "ETL ETROC", transform=ax.transAxes,
                    fontsize=10, style="italic", va="bottom")
            ax.text(1.0, 1.025, f"{identity}-{title} Histo", transform=ax.transAxes,
                    fontsize=9, ha="right", va="bottom")
            ax.text(1.0, 1.00, format_scan_time(results), transform=ax.transAxes,
                    fontsize=9, ha="right", va="bottom")
            ax.set_xlabel(f"{title} (DAC code)")
            ax.set_ylabel("Pixel count")
            if key == "baseline":
                bins = np.linspace(0, 1023, 65)
                ax.set_xlim(0, 1023)
                ax.set_xticks([0, 128, 256, 384, 512, 640, 768, 896, 1023])
            else:
                bins = np.arange(-0.5, 18, 1)
                ax.set_xlim(0, 17)
                ax.set_xticks(range(18))
            ax.xaxis.set_major_formatter(FormatStrFormatter("%.0f"))
            ax.yaxis.set_major_locator(MaxNLocator(integer=True))
            if values.size:
                # ax.hist(values, bins=bins, edgecolor="black", alpha=0.8)
                ax.hist(values, bins=bins, histtype="step", color="#7ec8e3", linewidth=1.3)
                ax.text(0.97, 0.95,
                        # f"Mean: {values.mean():.2f}\nStd: {values.std():.2f}",
                        # transform=ax.transAxes, ha="right", va="top")
                        f"Mean: {values.mean():.2f}, Std: {values.std():.2f}",
                        transform=ax.transAxes, ha="right", va="top")
            else:
                ax.text(0.5, 0.5, "No valid results", transform=ax.transAxes,
                        ha="center", va="center")
            # ax.grid(axis="y", alpha=0.25)
        figures.append(fig)
        plt.show()
    return figures


# This cell uses in-memory results and does not save image files.
blnw_figures = plot_blnw_maps(blnw_results)

blnw_histogram_figures = plot_blnw_histograms(blnw_results)

## 7. Enable only selected data links and check lock status - hardware
This step checks I2C again before enabling links. It does not run bitslip automatically. On a lock-check failure it attempts to mask all declared banks again; a masking failure is also reported. A lock check does not demonstrate valid payload reception.

In [ ]:
basic_checks_passed = False
if tuple(ids) != initialized_ids or list(chips) != ids:
    raise RuntimeError("Selection changed or initialization is incomplete")
if not etroc_register_checks_done or not etroc_power_checks_done:
    raise RuntimeError("Complete ETROC register and power configuration before enabling links")
if not blnw_scan_passed:
    raise RuntimeError("Complete the BL/NW scan successfully before enabling links")
try:
    connection_results = check_connections(chips)
    if not all(result["connected"] for result in connection_results.values()):
        raise RuntimeError(f"I2C checks failed: {connection_results}")
    applied_masks = configure_readout_links(rb, mapping, ids)
    rb.rerun_bitslip()
    link_results = check_links(rb, mapping, ids)
    for identity, result in link_results.items():
        for link in result["links"]:
            print(f"{identity}: {link['lpgbt']} E-link {link['elink']}, "
                  f"locked={link['locked']}, error={link['error']}")
    if not all(result["all_locked"] for result in link_results.values()):
        raise RuntimeError("One or more selected data links are not locked")
except Exception:
    try:
        configure_readout_links(rb, mapping, [])
    except Exception as cleanup_error:
        print(f"Failed to mask readout links: {cleanup_error}")
    raise

basic_checks_passed = True
show_status("Selected ETROCs respond and selected data links are locked", "success")
print("Only selected readout links are enabled; no acquisition has been started")

## 9. Configure DAC thresholds and activate scanned pixels - hardware

Applies `DAC = baseline + offset` to every pixel scanned in section 6d, reusing the baseline values already held in `blnw_results` rather than reloading from disk. Also writes the per-pixel activation registers (`workMode`, `enable_TDC`, `disDataReadout`, `disTrigPath`, `L1Adelay`) needed for a pixel to actually participate in readout, and opens the TOA/TOT/Cal trigger- and data-path threshold windows (`set_trigger_TH`, `set_data_TH`) to their full range. Those windows are a second filtering stage after the DAC comparator - a hit can pass the DAC threshold and still be dropped if it falls outside a narrow/default TOA-TOT-Cal window, independent of self-trigger vs. charge-injection mode. None of this is written anywhere earlier in this notebook. A pixel whose BL/NW scan failed (`baseline is None`) is skipped and reported, not silently assigned a threshold.

`reset_first=True` calls `etroc.reset()` per chip before applying thresholds. That reset also re-runs bitslip internally (see `ETROC.reset()` in the driver), so links may momentarily drop and relock. This cell does not itself re-verify link lock; rerun the check in section 7 afterward if you need to confirm the links are still locked.

In [ ]:
import time

THRESHOLD_OFFSET = 50   # DAC code; threshold = baseline + this offset
L1A_DELAY_DEFAULT = 0x01f5   # = 501 decimal; matches the L1Adelay readback in section 6a

# Full-range TOA/TOT/Cal trigger- and data-path windows: accept every hit
# instead of filtering by pulse shape. Narrow these later if you actually
# want TOA/TOT/Cal-based selection.
TOA_RANGE = (0, 0x3ff)
TOT_RANGE = (0, 0x1ff)
CAL_RANGE = (0, 0x3ff)

if not basic_checks_passed:
    raise RuntimeError("Complete section 7 (link enable) successfully before configuring thresholds")
if not blnw_scan_passed:
    raise RuntimeError("Complete the BL/NW scan before configuring thresholds")


def configure_thresholds(chips, blnw_results, offset=THRESHOLD_OFFSET,
                          l1a_delay=L1A_DELAY_DEFAULT, reset_first=True,
                          toa_range=TOA_RANGE, tot_range=TOT_RANGE, cal_range=CAL_RANGE):
    """Apply DAC threshold (baseline + offset), activate the pixels that
    were BL/NW-scanned in section 6d, and open their TOA/TOT/Cal trigger-
    and data-path windows. Reuses baseline values already in memory; does
    not reload from disk. A pixel with baseline=None (failed scan) is
    skipped, not assigned a threshold.

    reset_first=True calls etroc.reset() per chip, which internally also
    calls rb.rerun_bitslip() (see tamalero/ETROC.py). Re-run the link-lock
    check in section 7 afterward if you need to confirm links are still
    locked.
    """
    scan_pixels = blnw_results["pixels"]
    threshold_results = {}
    skipped_pixels = {}

    for identity, etroc in chips.items():
        if reset_first:
            etroc.reset()
            time.sleep(0.1)
            etroc.wr_reg("singlePort", 1)
            etroc.wr_reg("disDataReadout", 1, broadcast=True)
            etroc.wr_reg("QInjEn", 0, broadcast=True)
            etroc.wr_reg("enable_TDC", 0, broadcast=True)
            etroc.wr_reg("disTrigPath", 1, broadcast=True)
            etroc.wr_reg("workMode", 0, broadcast=True)

        applied = {}
        skipped = []
        for row, col in scan_pixels:
            baseline = blnw_results["baseline"][identity][(row, col)]
            if baseline is None:
                skipped.append((row, col))
                continue
            dac = baseline + offset

            etroc.wr_reg("workMode", 0, row=row, col=col, broadcast=False)
            etroc.wr_reg("enable_TDC", 1, row=row, col=col, broadcast=False)
            etroc.wr_reg("disDataReadout", 0, row=row, col=col, broadcast=False)
            etroc.wr_reg("disTrigPath", 0, row=row, col=col, broadcast=False)
            etroc.wr_reg("L1Adelay", l1a_delay, row=row, col=col, broadcast=False)
            etroc.wr_reg("DAC", dac, row=row, col=col, broadcast=False)

            etroc.set_trigger_TH("TOA", lower=toa_range[0], upper=toa_range[1], row=row, col=col, broadcast=False)
            etroc.set_trigger_TH("TOT", lower=tot_range[0], upper=tot_range[1], row=row, col=col, broadcast=False)
            etroc.set_trigger_TH("Cal", lower=cal_range[0], upper=cal_range[1], row=row, col=col, broadcast=False)
            etroc.set_data_TH("TOA", lower=toa_range[0], upper=toa_range[1], row=row, col=col, broadcast=False)
            etroc.set_data_TH("TOT", lower=tot_range[0], upper=tot_range[1], row=row, col=col, broadcast=False)
            etroc.set_data_TH("Cal", lower=cal_range[0], upper=cal_range[1], row=row, col=col, broadcast=False)

            applied[(row, col)] = {"baseline": baseline, "dac": dac}

        threshold_results[identity] = applied
        skipped_pixels[identity] = skipped
        level = "warning" if skipped else "success"
        show_status(f"{identity}: threshold applied to {len(applied)} pixels "
                    f"(offset={offset}), {len(skipped)} skipped (no baseline)", level)

    return threshold_results, skipped_pixels


threshold_results, threshold_skipped_pixels = configure_thresholds(chips, blnw_results)

## 10. Configure self-trigger - hardware

Enables the readout-board self-trigger path: pulses `LINK_RESET_PULSE`/`ECR_PULSE`/`BC0_PULSE`, then writes `TRIG_ENABLE_MASK`, `TRIG_DATA_SIZE` and `TRIG_DLY_SEL`. `TRIG_ENABLE_MASK` is independent from the data-readout mask configured in section 7 (`ETROC_DISABLE`/`ETROC_DISABLE_SLAVE`, per-elink): this mask is one bit per daughter-board position (bit0=board1 ... bit3=board4) and selects which board(s) actually generate the self-trigger, not which boards are read out. By default only the first selected board acts as the trigger source; pass `trigger_board_ids` to change that.

`trigger_delay` is written directly to `TRIG_DLY_SEL` via `kcu.write_node`, bypassing `ReadoutBoard.set_trigger_delay()`'s built-in 0-31 range check, which does not match real hardware usage (values around 466-472). The right value depends on the physical daughter-to-motherboard connection: a long FFC-cable path needs a different delay than a direct connection. Confirm which applies to the current setup before trusting the default.

In [ ]:
import time

if not basic_checks_passed:
    raise RuntimeError("Complete section 7 (link enable) successfully before configuring self-trigger")

TRIGGER_DELAY_DEFAULT = 466   # confirm against hardware; may differ for a long-FFC-cable daughter board


def configure_self_trigger(rb, mapping, ids, chips, trigger_delay=TRIGGER_DELAY_DEFAULT,
                            trigger_data_size=0x1, reset_pulses=True,
                            trigger_board_ids=None, trigger_combination_logic=0):
    """Readout-board-level self-trigger enable.

    trigger_board_ids: which board(s) generate the self-trigger, used to
    compute TRIG_ENABLE_MASK. Independent from `ids` (which boards are read
    out) - a board can be read out without being a trigger source. Defaults
    to only the first selected board.
    """

    if trigger_combination_logic not in (0, 1):
        raise ValueError("Trigger combination must be 0 (OR) or 1 (AND)")

    if trigger_board_ids is None:
        trigger_board_ids = [ids[0]]

    if reset_pulses:
        rb.kcu.write_node(f"READOUT_BOARD_{rb.rb}.LINK_RESET_PULSE", 0x1)
        rb.kcu.write_node(f"READOUT_BOARD_{rb.rb}.ECR_PULSE", 0x1)
        rb.kcu.write_node(f"READOUT_BOARD_{rb.rb}.BC0_PULSE", 0x1)

    trigger_enable_mask = build_trigger_mask(mapping, trigger_board_ids)

    rb.kcu.write_node(f"READOUT_BOARD_{rb.rb}.TRIG_ENABLE_MASK", trigger_enable_mask)
    rb.kcu.write_node(f"READOUT_BOARD_{rb.rb}.TRIG_COMBINATION_LOGIC", trigger_combination_logic)
    rb.kcu.write_node(f"READOUT_BOARD_{rb.rb}.TRIG_DATA_SIZE", trigger_data_size)
    rb.kcu.write_node(f"READOUT_BOARD_{rb.rb}.TRIG_DLY_SEL", trigger_delay)

    show_status(f"Trigger board(s): {trigger_board_ids}", "info")
    show_status(f"TRIG_ENABLE_MASK: 0x{trigger_enable_mask:X}", "info")
    show_status(f"TRIG_DATA_SIZE: {trigger_data_size}", "info")
    show_status(f"TRIG_DLY_SEL: {trigger_delay}", "info")
    time.sleep(0.1)

    for identity in ids:
        for bank, elinks in build_elinks(mapping, identity).items():
            for elink in elinks:
                locked = rb.etroc_locked(elink, slave=(bank == 1))
                print(f"{identity} elink {elink} (bank {bank}): locked={locked}")
                if not locked:
                    show_status(f"{identity} elink {elink} not locked, re-running bitslip", "warning")
                    rb.rerun_bitslip()
                    time.sleep(0.5)
                    locked = rb.etroc_locked(elink, slave=(bank == 1))
                    print(f"{identity} elink {elink}: locked={locked} (after re-lock)")
                    if not locked:
                        show_status(f"{identity} elink {elink} FAILED to lock", "error")

    show_status("Self-trigger configured", "success")
    for identity, etroc in chips.items():
        fc_status = etroc.FC_status()
        invalid_count = etroc.get_invalidFCCount()
        print(f"{identity}: FC status={fc_status}, invalid FC count={invalid_count}")


configure_self_trigger(rb, mapping, ids, chips, trigger_delay=TRIGGER_DELAY_DEFAULT, trigger_board_ids=trigger_ids, trigger_combination_logic=TRIGGER_COMBINATION_LOGIC,)

## 11. Charge injection (QInj) readout test - hardware

Before sending injection pulses, `enable_qinj_pixels()` writes the two QInj-specific per-pixel registers that section 9 does not touch: `QSel` (injected charge amplitude, `QSel = charge_fc - 1`) and `QInjEn` (1 = this pixel responds to injection pulses; the driver default is 0/off). Only the pixels scanned in section 6d with a valid baseline are enabled, matching `configure_thresholds()`. `CHARGE_FC_DEFAULT` below is a placeholder - confirm the actual injected-charge value used on this setup before trusting it.

`QInjEn` is a per-pixel state left set on the chip until explicitly cleared; it is not part of the self-trigger/cosmic configuration in section 10, so leaving it on for pixels not currently being used for QInj can make them respond to injection pulses unexpectedly in a later self-trigger run. `disable_qinj_pixels()` turns it back off; run it once you're done with charge injection on this selection.

`run_charge_injection_test()` sends `qinj_count` injection pulses per pixel through the FIFO and reads back the resulting events, checking the `ea` (event-assembly) integrity flag and looking for a header on every expected elink ("flashing bits", one per initialized ETROC). This assumes sections 9 and 10 (thresholds, self-trigger) and `enable_qinj_pixels()` above have already been run. It also does not switch `ELINK_WIDTH`/EPRX datarate to 320 Mbps the way the original QInj reference script does - this notebook stays at the 640 Mbps configured in section 4. If no events come back, that datarate mismatch is the first thing to check.

This does not save acquired data to disk; it only reports summary counts (`trigger_cnt`, `hit_counter`, `trailer_cnt`, `ea_clean`, `ea_issue`). Add file output separately if a permanent record is needed.

In [ ]:
import time
from datetime import datetime, timezone
from tamalero.FIFO import FIFO
from tamalero.DataFrame import DataFrame   # adjust the import path if DataFrame lives elsewhere in this driver

if not basic_checks_passed:
    raise RuntimeError("Complete section 7 (link enable) successfully before running charge injection")

QINJ_COUNT_DEFAULT = 1
EMPTY_SLOT_BCID = 500   # must be < 3563
CHARGE_FC_DEFAULT = 15   # placeholder - confirm the injected charge (fC) used on this setup


def enable_qinj_pixels(chips, blnw_results, charge_fc=CHARGE_FC_DEFAULT):
    """Enable charge injection on the pixels scanned in section 6d.

    QSel selects the injected charge amplitude (QSel = charge_fc - 1, per
    the reference confChip()). QInjEn=1 turns injection on for that pixel;
    it defaults to 0 (off). Only pixels with a valid baseline (see
    configure_thresholds()) are enabled.

    QInjEn stays set on the chip until cleared - call disable_qinj_pixels()
    once you are done with charge injection on this selection, so these
    pixels don't keep responding to injection pulses during a later
    self-trigger/cosmic run.
    """
    scan_pixels = blnw_results["pixels"]
    qsel = charge_fc - 1
    enabled = {}
    for identity, etroc in chips.items():
        count = 0
        for row, col in scan_pixels:
            baseline = blnw_results["baseline"][identity][(row, col)]
            if baseline is None:
                continue
            etroc.wr_reg("QSel", qsel, row=row, col=col, broadcast=False)
            etroc.wr_reg("QInjEn", 1, row=row, col=col, broadcast=False)
            count += 1
        enabled[identity] = count
        show_status(f"{identity}: QInj enabled on {count} pixels (QSel={qsel}, charge={charge_fc}fC)", "info")
    return enabled


def disable_qinj_pixels(chips, blnw_results):
    """Turn QInjEn back off on the pixels scanned in section 6d."""
    scan_pixels = blnw_results["pixels"]
    for identity, etroc in chips.items():
        for row, col in scan_pixels:
            etroc.wr_reg("QInjEn", 0, row=row, col=col, broadcast=False)
    show_status("QInj disabled on scanned pixels", "info")


qinj_enabled_pixels = enable_qinj_pixels(chips, blnw_results, charge_fc=CHARGE_FC_DEFAULT)


def run_charge_injection_test(chips, qinj_count=QINJ_COUNT_DEFAULT,
                               empty_slot_bcid=EMPTY_SLOT_BCID,
                               report_interval=10, max_iterations=1):
    """Self-triggered charge-injection (QInj) readout test.

    Assumes configure_thresholds() and configure_self_trigger() have already
    run (DAC/L1Adelay applied, TRIG_ENABLE_MASK/TRIG_DLY_SEL set, links
    locked). This function only sets emptySlotBCID, drives the FIFO and
    injection pulse, and reads back events.
    """
    df = DataFrame()
    fifo = FIFO(rb)
    fifo.reset()
    rb.reset_data_error_count()
    rb.enable_etroc_readout()
    rb.rerun_bitslip()
    fifo.use_etroc_data()
    time.sleep(0.1)

    rb.enable_etroc_trigger()
    time.sleep(0.1)

    for identity, etroc in chips.items():
        etroc.wr_reg("emptySlotBCID", empty_slot_bcid)

    trigger_cnt = 0
    hit_counter = 0
    trailer_cnt = 0
    ea_clean = 0
    ea_issue = 0

    start_time = datetime.now(timezone.utc)
    last_report_time = time.time()

    fifo.send_Qinj_only(count=qinj_count)  # per pixel

    rb.kcu.write_node(f"READOUT_BOARD_{rb.rb}.EVENT_CNT_RESET", 0x1)
    time.sleep(1)
    event_cnt = rb.kcu.read_node(f"READOUT_BOARD_{rb.rb}.EVENT_CNT").value()
    show_status(f"Event count in 1 second: {event_cnt}", "info")
    time.sleep(0.01)

    # Expected elinks computed from the mapping instead of hardcoded, so this
    # works regardless of which/how many boards are selected.
    expected_elinks = set()
    for identity in chips:
        for bank, elinks in build_elinks(mapping, identity).items():
            expected_elinks.update(elinks)

    for iteration in range(max_iterations):
        try:
            print(f"Occupancy: {fifo.get_occupancy()}")
            data = fifo.pretty_read(df)
            time.sleep(0.1)  # slow down DAQ to avoid uhal UDP errors

            if len(data) > 0:
                flashing_headers = {}
                for event in data:
                    if event and len(event) >= 2:
                        if event[0] == "header":
                            trigger_cnt += 1
                            if event[1]["bcid"] == "1":
                                flashing_headers[event[1]["elink"]] = event
                        elif event[0] == "data":
                            print(event)
                            if event[1]["ea"] == 0:
                                ea_clean += 1
                            else:
                                show_status("ea is not 0", "error")
                                ea_issue += 1
                            hit_counter += 1
                        elif event[0] == "trailer":
                            trailer_cnt += 1

                if expected_elinks.issubset(flashing_headers.keys()):
                    show_status(f"Flashing bit detected on all {len(expected_elinks)} expected elinks", "success")
                    for elink in sorted(flashing_headers.keys()):
                        print(f"  elink {elink}: {flashing_headers[elink]}")

            time.sleep(0.1)
            current_time = time.time()
            if current_time - last_report_time >= report_interval:
                elapsed = (datetime.now(timezone.utc) - start_time).total_seconds()
                print(f"\n--- Status Report ---")
                print(f"Running time: {elapsed:.1f} s")
                print(f"Total hits: {hit_counter}")
                print(f"Trigger count: {trigger_cnt}")
                print(f"Trailer count: {trailer_cnt}")
                last_report_time = current_time
            time.sleep(0.05)

        except Exception as exc:
            show_status(f"Data acquisition error: {exc}", "error")
            time.sleep(1)
            continue

    current_mask = rb.kcu.read_node(f"READOUT_BOARD_{rb.rb}.TRIG_ENABLE_MASK").value()
    show_status(f"Readback TRIG_ENABLE_MASK: 0x{current_mask:X}", "info")
    show_status(f"Total hits: {hit_counter}, trigger count: {trigger_cnt}, trailer count: {trailer_cnt}", "info")
    if hit_counter:
        show_status(f"ea clean: {ea_clean}, ea issue: {ea_issue}",
                    "success" if ea_issue == 0 else "warning")

    return {
        "trigger_cnt": trigger_cnt,
        "hit_counter": hit_counter,
        "trailer_cnt": trailer_cnt,
        "ea_clean": ea_clean,
        "ea_issue": ea_issue,
    }


qinj_results = run_charge_injection_test(chips, qinj_count=QINJ_COUNT_DEFAULT)

# Uncomment once you're done running charge injection on this selection,
# so these pixels stop responding to injection pulses in later runs:
# disable_qinj_pixels(chips, blnw_results)

## 12. End the basic check - hardware
Mask all declared readout banks. This does not clear FIFO contents, stop external trigger sources, disable injection registers or power down chips. When injection/acquisition is added later, its stop procedure must be added explicitly.

Running every cell includes this final masking step. To inspect enabled links, or the threshold/self-trigger/charge-injection sections above, pause before this cell, then run it when finished.

In [ ]:
configure_readout_links(rb, mapping, [])
print(f"Basic checks passed: {basic_checks_passed}")
show_status("All declared readout banks are masked", "info")

## 13. Adapt the selected chips for the independent WS test

Only the object container is adapted: chips[identity] becomes (etroc, identity, []). The original test ignores the third tuple item. All selected ETROCs are passed to the original loop; there is still only one sampler at its fixed address.

In [ ]:
# Adapt selected ETROC objects to the tuple format used by the original test.
# This cell performs no hardware writes.
if not ids or tuple(ids) != initialized_ids or list(chips) != ids:
    raise RuntimeError("Selection changed or ETROC initialization is incomplete")
if not etroc_register_checks_done or not etroc_power_checks_done:
    raise RuntimeError("Complete ETROC register and power configuration first")

etroc_configs = [
    (chips[identity], identity, [])
    for identity in ids
]
print("Waveform test ETROCs:", [item[1] for item in etroc_configs])
print("Waveform outputs:", (Path.cwd() / "waveform_data").resolve())


## 14. Original waveform sampler test - unchanged

Execute explicitly after checking the selection above. Register values, pulse sequence, delays, decoding, retries, save behavior and plotting are exactly as supplied. No extra cleanup or automatic pulse has been inserted.


In [ ]:
import time
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
from tamalero.colors import red, green

print("\n" + "=" * 70)
print("  WAVEFORM SAMPLER TEST (KCU105 with WS_START_PULSE)")
print("=" * 70)

WS_I2C_ADDRESS = 0x43
WS_I2C_CHANNEL = 1
WS_ADR_NBYTES = 1
WS_TEST_PIXELS = [(0, 14)]
WS_OFFSET = 10
WS_NUM_SAMPLES = 1
OUTPUT_DIR = Path("./waveform_data")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

class TamaleroWaveformSampler:
    def __init__(self, rb, ws_addr=0x40, i2c_master=1):
        self.rb = rb
        self.lpgbt = rb.DAQ_LPGBT
        self.kcu = rb.kcu
        self.ws_addr = ws_addr
        self.i2c_master = i2c_master
        self.adr_nbytes = 1

    def i2c_write(self, reg, val):
        self.lpgbt.I2C_write(reg=reg, val=val, master=self.i2c_master,
                             slave_addr=self.ws_addr, adr_nbytes=self.adr_nbytes)

    def i2c_read(self, reg, nbytes=1):
        return self.lpgbt.I2C_read(reg=reg, master=self.i2c_master,
                                   slave_addr=self.ws_addr, nbytes=nbytes, adr_nbytes=self.adr_nbytes)

    def modify_reg_bit(self, reg, bit_mask, val_bool):
        current = self.i2c_read(reg)
        if isinstance(current, list):
            current = current[0]
        if val_bool:
            new_val = current | bit_mask
        else:
            new_val = current & (~bit_mask)
        self.i2c_write(reg, new_val)

    def init_ws(self):
        print(f"  [{hex(self.ws_addr)}] Initializing WS Global Config...")
        self.i2c_write(0x1F, 0x2B)
        self.i2c_write(0x0F, 0x00)
        self.i2c_write(0x0E, 0x00)
        self.i2c_write(0x0D, 0x10)

    def arm_ws(self):
        print(f"  [{hex(self.ws_addr)}] Configing WS Memory Config...")
        self.i2c_write(0x1F, 0x22)
        self.i2c_write(0x1F, 0x0B)

    def trigger_ws_start(self):
        print(f"  [{hex(self.ws_addr)}] Sending WS_START_PULSE via KCU...")
        self.kcu.write_node(f"READOUT_BOARD_{self.rb.rb}.WS_START_PULSE", 0x1)

    def trigger_ws_stop(self):
        print(f"  [{hex(self.ws_addr)}] Sending WS_STOP_PULSE via KCU...")
        self.kcu.write_node(f"READOUT_BOARD_{self.rb.rb}.WS_STOP_PULSE", 0x1)

    def stop_and_close_ws(self):
        self.i2c_write(0x1F, 0x09)

    def read_data(self):
        print(f"  [{hex(self.ws_addr)}] Reading 1024 words from WS RAM...")
        self.modify_reg_bit(0x1F, 0b00000100, True)

        max_steps = 1024
        base_data = []
        coeff = 0.085
        time_coeff = 0.390625

        for address in tqdm(range(max_steps), desc="  Reading WS", ncols=80):
            rd_add_MSB = address // 4
            rd_add_LSB = (address % 4) * 64

            self.i2c_write(0x1D, rd_add_MSB)
            self.i2c_write(0x1C, rd_add_LSB)

            tmp_data = None
            for _ in range(3):
                tmp_data = self.i2c_read(0x20, nbytes=2)
                if tmp_data is not None and len(tmp_data) >= 2:
                    break
                time.sleep(0.005)

            if tmp_data is None or len(tmp_data) < 2:
                print(red(f"\n  I2C Read Failed at Address {address}, skipping..."))
                continue

            data_int = (tmp_data[0] >> 2) | (tmp_data[1] << 6)
            pointer = (data_int >> 13) & 1
            Dout_S1 = (data_int >> 7) & 0x3F

            Dout_S2 = (((data_int >> 6) & 1) * 24 +
                       ((data_int >> 5) & 1) * 16 +
                       ((data_int >> 4) & 1) * 10 +
                       ((data_int >> 3) & 1) * 6 +
                       ((data_int >> 2) & 1) * 4 +
                       ((data_int >> 1) & 1) * 2 +
                        (data_int & 1))

            base_data.append({
                "Address": address,
                "Data": data_int,
                "Raw": f"{data_int:014b}",
                "pointer": pointer,
                "Dout_S1": Dout_S1,
                "Dout_S2": Dout_S2,
                "Dout": Dout_S1 - (coeff * Dout_S2),
            })

        df = pd.DataFrame(base_data)
        if len(df) == 0:
            self.modify_reg_bit(0x1F, 0b00000100, False)
            return df

        channels = 8
        steps_per_ch = len(df) // channels

        df['Channel'] = np.repeat(np.arange(1, channels + 1), steps_per_ch)
        df['Step'] = np.tile(np.arange(steps_per_ch), channels)

        pointer_mask = (df['Channel'] == channels) & (df['pointer'] != 0)
        if pointer_mask.any():
            pointer_step = df.loc[pointer_mask, 'Step'].iloc[0]
            df['Step'] = (df['Step'] - (pointer_step + 1)) % steps_per_ch

        df['Time_Index'] = df['Step'] * channels + (channels - df['Channel'])
        df['Time_ns'] = df['Time_Index'] * time_coeff
        df = df.sort_values('Time_Index').reset_index(drop=True)

        self.modify_reg_bit(0x1F, 0b00000100, False)
        return df

print("\n1. Initializing Waveform Sampler...")
ws = TamaleroWaveformSampler(rb, ws_addr=WS_I2C_ADDRESS, i2c_master=WS_I2C_CHANNEL)

print("  Testing I2C connection...")
try:
    val = ws.i2c_read(0x00)
    if isinstance(val, list):
        val = val[0]
    print(green(f"  WS connected! Reg 0x00 = 0x{val:02X}"))
except Exception as e:
    print(red(f"  WS connection failed: {e}"))

ws.init_ws()

print(f"\n2. Configuring ETROC test pixels: {WS_TEST_PIXELS}")

for etroc, chip_name, _ in etroc_configs:
    for row, col in WS_TEST_PIXELS:
        baseline, nw = etroc.auto_threshold_scan(row=row, col=col, broadcast=False, offset=1023, use=False, verbose=False)
        print(f"  {chip_name} ({row},{col}): baseline={baseline}, NW={nw}")

        etroc.wr_reg("workMode", 0, row=row, col=col, broadcast=False)
        etroc.wr_reg("enable_TDC", 1, row=row, col=col, broadcast=False)
        etroc.wr_reg("disDataReadout", 0, row=row, col=col, broadcast=False)
        etroc.wr_reg("disTrigPath", 0, row=row, col=col, broadcast=False)
        etroc.wr_reg("DAC", baseline + WS_OFFSET, row=row, col=col, broadcast=False)
        etroc.wr_reg("QSel", 0x1E, row=row, col=col, broadcast=False)
        etroc.wr_reg("QInjEn", 1, row=row, col=col, broadcast=False)
        etroc.wr_reg("RfSel", 0x00, row=row, col=col, broadcast=False)

print(green("  ETROC pixels configured"))

print(f"\n3. Collecting {WS_NUM_SAMPLES} Waveform samples...")

all_waveforms = []

for sample_idx in range(WS_NUM_SAMPLES):
    print(f"\n  === Sample {sample_idx + 1}/{WS_NUM_SAMPLES} ===")

    try:
        ws.arm_ws()
        time.sleep(0.1)

        # print("  Sending QInj...")
        # fifo.send_Qinj_only()
        # # time.sleep(0.1)

        # ws.trigger_ws_start()
        # time.sleep(0.1)


        ws.trigger_ws_stop()
        time.sleep(0.1)


        df = ws.read_data()
        
        if len(df) > 0:
            df['sample_idx'] = sample_idx
            df['timestamp'] = datetime.now().isoformat()
            all_waveforms.append(df)

            print(f"  Got {len(df)} points")
            print(f"  Dout_S1 range: [{df['Dout_S1'].min()}, {df['Dout_S1'].max()}]")
            print(f"  Dout range: [{df['Dout'].min():.2f}, {df['Dout'].max():.2f}]")
        else:
            print(red("  Data frame is empty!"))

        ws.stop_and_close_ws()

    except Exception as e:
        print(red(f"  Error: {e}"))
        import traceback
        traceback.print_exc()

    time.sleep(0.1)

print(f"\n4. Saving results...")

if all_waveforms:
    df_all = pd.concat(all_waveforms, ignore_index=True)
    csv_filename = OUTPUT_DIR / f"waveform_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    df_all.to_csv(csv_filename, index=False)
    print(green(f"  Saved to: {csv_filename}"))

    print(f"\n  === Data Summary ===")
    print(f"  Total samples: {len(all_waveforms)}")
    print(f"  Points per sample: {len(all_waveforms[0])}")
    print(f"  Dout mean: {df_all['Dout'].mean():.3f}")
    print(f"  Dout std:  {df_all['Dout'].std():.3f}")
    print(f"  Dout min:  {df_all['Dout'].min():.3f}")
    print(f"  Dout max:  {df_all['Dout'].max():.3f}")

    print(f"\n5. Plotting waveforms...")
    # fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig, axes = plt.subplots()

    # ax1 = axes[0, 0]
    ax1 = axes

    for i, df in enumerate(all_waveforms):
        ax1.plot(df['Time_ns'], df['Dout'], alpha=0.7, label=f'Sample {i+1}')
    ax1.set_xlabel('Time [ns]')
    ax1.set_ylabel('Dout')
    ax1.set_title('ET2.03-Bare30 Waveforms w/ QInj (Time Domain)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    # ax1.set_xlim([30, 380])
    # ax1.set_ylim([30, 40])

    # ax2 = axes[0, 1]
    # df_mean = df_all.groupby('Time_ns')['Dout'].mean().reset_index()
    # df_std = df_all.groupby('Time_ns')['Dout'].std().reset_index()
    # ax2.plot(df_mean['Time_ns'], df_mean['Dout'], 'b-', linewidth=2, label='Mean')
    # ax2.fill_between(df_mean['Time_ns'],
    #                  df_mean['Dout'] - df_std['Dout'],
    #                  df_mean['Dout'] + df_std['Dout'],
    #                  alpha=0.3, label='±1σ')
    # ax2.set_xlabel('Time [ns]')
    # ax2.set_ylabel('Dout')
    # ax2.set_title('Average Waveform')
    # ax2.legend()
    # ax2.grid(True, alpha=0.3)

    # ax3 = axes[1, 0]
    # df_first = all_waveforms[0]
    # for ch in range(1, 9):
    #     ch_data = df_first[df_first['Channel'] == ch]
    #     ax3.plot(ch_data['Time_ns'], ch_data['Dout'], label=f'Ch {ch}')
    # ax3.set_xlabel('Time [ns]')
    # ax3.set_ylabel('Dout')
    # ax3.set_title('Waveform by Channel (Sample 1)')
    # ax3.legend(ncol=2)
    # ax3.grid(True, alpha=0.3)

    # ax4 = axes[1, 1]
    # ax4.hist(df_all['Dout'], bins=50, edgecolor='black', alpha=0.7)
    # ax4.axvline(df_all['Dout'].mean(), color='r', linestyle='--', label=f"Mean: {df_all['Dout'].mean():.2f}")
    # ax4.set_xlabel('Dout')
    # ax4.set_ylabel('Count')
    # ax4.set_title('Dout Distribution')
    # ax4.legend()
    # ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    fig_filename = OUTPUT_DIR / f"waveform_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
    plt.savefig(fig_filename, dpi=150)
    print(green(f"  Figure saved to: {fig_filename}"))
    plt.show()

else:
    print(red("  No data to plot!"))

print("\n" + "=" * 70)
print("  WAVEFORM SAMPLER TEST COMPLETE")
print("=" * 70)



## 15. Offline waveform DSP and comparison

This cell only reads a CSV and saves a comparison figure; it does not access hardware. It can run independently with an explicit DSP_CSV_PATH. Requires pandas, numpy, matplotlib and scipy. The original acquisition cell is unchanged.

All samples are processed separately; at most four are plotted. The original acquisition defaults to one sample, so one curve is expected unless the source CSV contains more samples. The 40-80 ns pedestal window, median kernel 5, Savitzky-Golay window 11/order 3, axis limits and 150 ns reference marker are retained. Filtering can change pulse shape; the raw CSV remains unchanged. The reference marker does not confirm an injection occurred.

Set DISPLAY_MODE="both" to show and optionally save a raw figure plus the two-panel DSP comparison. Set DISPLAY_MODE="raw" to skip filtering entirely (SciPy is not required in this mode). SAVE_ANALYSIS_FIGURES=False displays these figures without saving them. The standalone raw figure uses unchanged Dout values; the comparison raw panel retains the supplied baseline subtraction. Acquisition figures and CSV saving are unchanged.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Leave as None to process the CSV saved by the acquisition cell above.
# Set an explicit path to process an existing file without running hardware cells.
DSP_CSV_PATH = None
# "both": raw figure and DSP comparison; "raw": raw figure only, no filtering.
DISPLAY_MODE = "both"
# Applies only to this offline section; acquisition save behavior is unchanged.
SAVE_ANALYSIS_FIGURES = True
if DISPLAY_MODE not in {"both", "raw"}:
    raise ValueError("DISPLAY_MODE must be 'both' or 'raw'")
# DSP_CSV_PATH = Path("/absolute/path/to/waveform_20260515_140804.csv")

if DSP_CSV_PATH is None:
    if not globals().get("all_waveforms") or "csv_filename" not in globals():
        raise RuntimeError("Run a successful acquisition or set DSP_CSV_PATH")
    file_path = Path(csv_filename)
else:
    file_path = Path(DSP_CSV_PATH).expanduser()

df_raw = pd.read_csv(file_path)
required_columns = {"sample_idx", "Time_ns", "Channel", "Dout"}
missing_columns = required_columns - set(df_raw.columns)
if missing_columns:
    raise ValueError(f"Missing CSV columns: {sorted(missing_columns)}")
if df_raw.empty or df_raw[list(required_columns)].isna().any().any():
    raise ValueError("Waveform data is empty or contains missing required values")


def comprehensive_dsp_pipeline(df):
    processed_samples = []
    # Process each sample separately and retain the full time range.
    for idx, group in df.groupby('sample_idx'):
        sample = group.sort_values('Time_ns').copy()
        if len(sample) < 11:
            raise ValueError(f"Sample {idx} needs at least 11 points")
        if not sample['Channel'].isin(range(1, 9)).all():
            raise ValueError(f"Sample {idx} contains channels outside 1..8")

        # Align each interleaved ADC pedestal using the 40-80 ns window.
        for ch in range(1, 9):
            ch_mask = sample['Channel'] == ch
            quiet_zone = (sample['Time_ns'] > 40) & (sample['Time_ns'] < 80) & ch_mask
            if quiet_zone.any():
                channel_baseline = sample.loc[quiet_zone, 'Dout'].mean()
                sample.loc[ch_mask, 'Dout_adj'] = sample.loc[ch_mask, 'Dout'] - channel_baseline
            else:
                sample.loc[ch_mask, 'Dout_adj'] = sample.loc[ch_mask, 'Dout'] - sample.loc[ch_mask, 'Dout'].mean()

        # Preserve the supplied median and Savitzky-Golay parameters.
        sample['Dout_med'] = medfilt(sample['Dout_adj'], kernel_size=5)
        sample['Dout_smooth'] = savgol_filter(sample['Dout_med'], window_length=11, polyorder=3)
        processed_samples.append(sample)
    return processed_samples


# Plot the original ADC values without baseline subtraction or filtering.
fig_raw, ax_raw = plt.subplots(figsize=(14, 5))
for i, (idx, group) in enumerate(df_raw.groupby('sample_idx')):
    if i >= 4:
        break
    group = group.sort_values('Time_ns')
    ax_raw.plot(group['Time_ns'], group['Dout'], alpha=0.7,
                label=f'Raw Sample {idx+1}')
ax_raw.set_title('Original Raw Waveforms (Unprocessed)')
ax_raw.set_xlabel('Time [ns]')
ax_raw.set_ylabel('Amplitude [Dout]')
ax_raw.grid(True, linestyle=':', alpha=0.5)
ax_raw.legend()
fig_raw.tight_layout()
raw_figure_path = file_path.with_name(file_path.stem + '_Raw.png')
if SAVE_ANALYSIS_FIGURES:
    fig_raw.savefig(raw_figure_path, dpi=200)
    print(f"Raw figure saved: {raw_figure_path.resolve()}")
plt.show()

# Clear processed results from any previous execution in this kernel.
all_processed = None
if DISPLAY_MODE == "both":
    from scipy.signal import medfilt, savgol_filter
    all_processed = comprehensive_dsp_pipeline(df_raw)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
    colors = plt.cm.tab10(np.linspace(0, 0.5, 4))
    display_count = min(4, len(all_processed))

    for i, (idx, group) in enumerate(df_raw.groupby('sample_idx')):
        if i >= 4:
            break
        raw_offset = group[group['Time_ns'] > 40]['Dout'].mean()
        ax1.plot(group['Time_ns'], group['Dout'] - raw_offset,
                 color=colors[i], alpha=0.5, label=f'Raw Sample {idx+1}')

    ax1.set_title(f'{display_count} Original Raw Waveforms (with Spikes & Channel Mismatch)', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Amplitude [Dout]')
    ax1.set_ylim([-15, 20])
    ax1.grid(True, linestyle=':', alpha=0.5)
    ax1.legend(loc='upper right', ncol=2)

    for i, df_p in enumerate(all_processed):
        if i >= 4:
            break
        idx = df_p['sample_idx'].iloc[0]
        ax2.plot(df_p['Time_ns'], df_p['Dout_smooth'], color=colors[i],
                 linewidth=2.5, label=f'Cleaned Sample {idx+1}')

    ax2.set_title(f'{display_count} Processed Waveforms (Cleaned & Aligned)', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Time [ns]')
    ax2.set_ylabel('Amplitude [Normalized]')
    ax2.set_ylim([-6, 8])
    ax2.grid(True, linestyle='--', alpha=0.5)
    ax2.legend(loc='upper right', ncol=2)

    # This is the supplied reference marker, not a measured injection time.
    for ax in [ax1, ax2]:
        ax.axvline(x=150, color='red', linestyle='--', alpha=0.3)
        ax.text(155, ax.get_ylim()[1]*0.7, 'QInj Expected', color='red', alpha=0.6)

    plt.tight_layout()
    # Keep the comparison beside its source CSV without overwriting raw data.
    dsp_figure_path = file_path.with_name(file_path.stem + '_Full_Length_DSP_Study.png')
    if SAVE_ANALYSIS_FIGURES:
        plt.savefig(dsp_figure_path, dpi=200)
    plt.show()
    if SAVE_ANALYSIS_FIGURES:
        print(f"Comparison figure saved: {dsp_figure_path.resolve()}")
